In [ ]:
from pyspark.sql import SparkSession, functions as sf

import labtech
from labtech.runners import ThreadRunnerBackend

In [ ]:
# Ensure you first run: make spark-cluster
spark = (
    SparkSession.builder
    # A remote connection is sufficient for this example,
    # only requiring pyspark-client to be pip installed.
    .remote('sc://localhost:15002')
    .appName('labtech_demo')
    .getOrCreate()
)

In [ ]:
@labtech.task
class Experiment:
    table: str
    power: int

    def run(self):
        labtech.logger.info(f'Running with table {self.table} and power {self.power}')
        return (
            spark.read.table(self.table)
            .withColumn('raised_value', sf.col('value') ** self.power)
            .select(sf.mean(sf.col('raised_value')).alias('mean'))
            # We run collect() so that the Spark execution of transformations
            # is triggered from the task's thread. Other task threads will be
            # free to run while this thread waits for Spark to finish executing.
            .collect()[0]['mean']
        )


# Prepare a DataFrame in Spark that can be referenced by name from each task.
table_name = 'dataset'
spark_df = spark.createDataFrame([
    {'value': value} for value in range(1000)
])
spark_df.createOrReplaceTempView(table_name)

experiments = [
    Experiment(
        table=table_name,
        power=power,
    )
    for power in range(10)
]

lab = labtech.Lab(
    # Because our experiment tasks delegate execution to the Spark cluster we can
    # just start tasks concurrently from separate lightweight threads.
    runner_backend=ThreadRunnerBackend(),
    # Max workers should be set relative to available Spark cores
    # and the number of cores that each task can leverage.
    max_workers=3,
    storage='storage/spark_lab',
)

cached_experiments = lab.cached_tasks([Experiment])
print(f'Clearing {len(cached_experiments)} cached experiments.')
lab.uncache_tasks(cached_experiments)

results = lab.run_tasks(experiments)
print(results)